# UFO Sightings: Two Altair Visualizations

Data: NUFORC UFO sighting reports (geocoded, time-standardized), read directly from the course data repository.

This notebook builds two plots:
1. **Sightings over time, linked to reported shape** – drag on the timeline to filter the shape bar chart.
2. **US map of sightings by state** – use the dropdown to show only one reported shape.

In [ ]:
import pandas as pd
import altair as alt

## 1. Read the data from a URL
The CSV has no header row, so the column names are supplied here.

In [ ]:
ufo_url = "https://github.com/UIUC-iSchool-DataViz/is445_data/raw/main/ufo-scrubbed-geocoded-time-standardized-00.csv"

column_names = ["date", "city", "state", "country", "shape", "duration_seconds",
                "duration_reported", "description", "report_date", "latitude", "longitude"]

ufo = pd.read_csv(ufo_url, header=None, names=column_names, low_memory=False)
print(ufo.shape)
ufo.head()

## 2. Clean and transform

* Parse the date strings into timestamps in Pandas (easier than doing it in Altair). A few rows use times like `24:00`, which cannot be parsed; those become `NaT` and are dropped.
* Clean the `shape` text and keep the 8 most common shapes; every other shape (including the original "other" and "unknown") is grouped as **other**.

In [ ]:
ufo["date"] = pd.to_datetime(ufo["date"], format="%m/%d/%Y %H:%M", errors="coerce")
rows_before = len(ufo)
ufo = ufo.dropna(subset=["date"]).copy()
print(f"Dropped {rows_before - len(ufo)} rows with unparseable dates")

ufo["year"] = ufo["date"].dt.year

ufo["shape"] = ufo["shape"].fillna("unknown").astype(str).str.strip().str.lower()
named_shapes = ufo.loc[~ufo["shape"].isin(["other", "unknown"]), "shape"]
top_shapes = named_shapes.value_counts().head(8).index.tolist()
ufo["shape_group"] = ufo["shape"].where(ufo["shape"].isin(top_shapes), "other")
shape_list = top_shapes + ["other"]
print(shape_list)

## 3. Plot 1 – Sightings over time, linked to reported shape

Data transformation: count sightings per **year × shape group** (from 1950 on, since earlier years have very few reports). Aggregating in Pandas keeps the embedded data small.

Interactivity: an **interval brush** on the timeline filters the bar chart below it.

In [ ]:
trend = (ufo[ufo["year"] >= 1950]
         .groupby(["year", "shape_group"])
         .size()
         .reset_index(name="sightings"))
trend.head()

In [ ]:
brush = alt.selection_interval(encodings=["x"])

timeline = alt.Chart(trend).mark_area(
    line={"color": "#2c6e91"}, color="#9ecae1", opacity=0.8
).encode(
    x=alt.X("year:Q", title="Year", axis=alt.Axis(format="d")),
    y=alt.Y("sum(sightings):Q", title="Sightings per year"),
    tooltip=[alt.Tooltip("year:Q", title="Year", format="d"),
             alt.Tooltip("sum(sightings):Q", title="Sightings")]
).add_params(
    brush
).properties(
    width=600, height=220,
    title=alt.TitleParams("UFO sightings per year",
                          subtitle="Drag across the chart to select a range of years")
)

shape_bars = alt.Chart(trend).mark_bar(color="#2c6e91").encode(
    x=alt.X("sum(sightings):Q", title="Sightings in selected years"),
    y=alt.Y("shape_group:N", sort="-x", title="Reported shape"),
    tooltip=[alt.Tooltip("shape_group:N", title="Shape"),
             alt.Tooltip("sum(sightings):Q", title="Sightings")]
).transform_filter(
    brush
).properties(
    width=600, height=220,
    title="Most common reported shapes (in selected years)"
)

plot1 = alt.vconcat(timeline, shape_bars)
plot1

## 4. Plot 2 – US map of sightings by state

Data transformation: keep US reports only, count sightings per **state × shape group**, and add each state's FIPS code so the counts can be joined to the US states map (TopoJSON from vega-datasets).

Interactivity: a **dropdown** filters the map to one reported shape; hovering shows the state and its count.

In [ ]:
state_fips = {
    "al": 1, "ak": 2, "az": 4, "ar": 5, "ca": 6, "co": 8, "ct": 9, "de": 10, "dc": 11,
    "fl": 12, "ga": 13, "hi": 15, "id": 16, "il": 17, "in": 18, "ia": 19, "ks": 20,
    "ky": 21, "la": 22, "me": 23, "md": 24, "ma": 25, "mi": 26, "mn": 27, "ms": 28,
    "mo": 29, "mt": 30, "ne": 31, "nv": 32, "nh": 33, "nj": 34, "nm": 35, "ny": 36,
    "nc": 37, "nd": 38, "oh": 39, "ok": 40, "or": 41, "pa": 42, "ri": 44, "sc": 45,
    "sd": 46, "tn": 47, "tx": 48, "ut": 49, "vt": 50, "va": 51, "wa": 53, "wv": 54,
    "wi": 55, "wy": 56,
}

us = ufo[(ufo["country"] == "us") & (ufo["state"].isin(state_fips))].copy()

state_counts = (us.groupby(["state", "shape_group"])
                  .size()
                  .reset_index(name="sightings"))
state_counts["id"] = state_counts["state"].map(state_fips)
state_counts["state"] = state_counts["state"].str.upper()
state_counts.head()

In [ ]:
states = alt.topo_feature(
    "https://cdn.jsdelivr.net/npm/vega-datasets@v1.29.0/data/us-10m.json", "states")

shape_dropdown = alt.binding_select(
    options=[None] + shape_list,
    labels=["all shapes"] + shape_list,
    name="Reported shape: ")
shape_select = alt.selection_point(fields=["shape_group"], bind=shape_dropdown)

background = alt.Chart(states).mark_geoshape(fill="#e5e5e5", stroke="white")

choropleth = alt.Chart(state_counts).mark_geoshape(stroke="white").encode(
    shape="geo:G",
    color=alt.Color("total:Q",
                    title="Sightings",
                    scale=alt.Scale(type="sqrt", scheme="viridis")),
    tooltip=[alt.Tooltip("state:N", title="State"),
             alt.Tooltip("total:Q", title="Sightings")]
).add_params(
    shape_select
).transform_filter(
    shape_select
).transform_aggregate(
    total="sum(sightings)", groupby=["id", "state"]
).transform_lookup(
    lookup="id",
    from_=alt.LookupData(data=states, key="id"),
    as_="geo"
)

plot2 = (background + choropleth).project(type="albersUsa").properties(
    width=650, height=400,
    title=alt.TitleParams("UFO sightings by US state",
                          subtitle="Use the dropdown to show a single reported shape")
)
plot2

## 5. Export for the Jekyll page
The JSON files are embedded in the website with vega-embed; the HTML files are for quick local checks.

In [ ]:
plot1.save("ufo_trend_by_shape.json")
plot2.save("ufo_map_by_state.json")
plot1.save("ufo_trend_by_shape.html")
plot2.save("ufo_map_by_state.html")